In [76]:
# Set Up 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


import os

pd.set_option('display.max_columns', None)  # Show all columns in DataFrame display

In [77]:
import pandas as pd
DAY_SENSOR_DATA_PATH = '../Data/RawData/Trial_2_Day_Sensor_Data.xlsx'
NIGHT_SENSOR_DATA_PATH = '../Data/RawData/Trial_2_Night_Sensor_Data.xlsx'
NUTRIENT_TEMP_SENSOR_DATA_PATH = '../Data/RawData/Trial_2_Nutrient_Temperature_Sensor_Data.xlsx' # atlas pt 1000 missing from other sensor data
LETTUCE_FW_DATA_PATH = '../Data/RawData/Trial_2_Lettuce_FW_Data_Per_Plant.csv'

def load_day_sensor_data():
    return pd.read_excel(DAY_SENSOR_DATA_PATH)

def load_night_sensor_data():
    return pd.read_excel(NIGHT_SENSOR_DATA_PATH)

def load_nutrient_temp_sensor_data():
    return pd.read_excel(NUTRIENT_TEMP_SENSOR_DATA_PATH)

def load_lettuce_fw_data():
    return pd.read_csv(LETTUCE_FW_DATA_PATH)


In [78]:
# features we are keeping 
# #    'carbon_dioxide': 'blue',
#     'temp_env': 'orange',
#     'humidity': 'green',
#     'pressure': 'red',
#     'electrical_conductivity': 'purple',
#     'volume': 'brown',
#     'temp_nutr': 'pink',
#     'volume_flow_rate': 'gray',
#     'weight_lag_1': 'cyan',
#     'weight_lag_2': 'magenta'

In [79]:
# combine day and night data 
day_sensor_data = load_day_sensor_data()
night_sensor_data = load_night_sensor_data()




# add nutrient temperature data to rest of sensor data (waiting on more data from Sam)
nutrient_temp_sensor_data = load_nutrient_temp_sensor_data()

# add day column to sensor data and average sensor data across 4 days



In [80]:
day_sensor_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7894 entries, 0 to 7893
Data columns (total 7 columns):
 #   Column                                                     Non-Null Count  Dtype         
---  ------                                                     --------------  -----         
 0   DateTime                                                   7894 non-null   datetime64[ns]
 1   Atlas pH (CH0, Ion Concentration, pH)                      6865 non-null   float64       
 2   Atlas CO2 (Carbon Dioxide Gas) (CH0, Carbon Dioxide, ppm)  5652 non-null   float64       
 3   Atlas EC (CH0, Electrical Conductivity, μS/cm)             7580 non-null   float64       
 4   BME280 (CH0, Temperature, °C)                              4956 non-null   float64       
 5   BME280 (CH1, Humidity, %)                                  5718 non-null   float64       
 6   BME280 (CH5, Vapor Pressure Deficit, Pa)                   4429 non-null   float64       
dtypes: datetime64[ns](1), float64(6)
m

In [81]:
day_sensor_data['DateTime'].min(), day_sensor_data['DateTime'].max()

(Timestamp('2026-01-19 19:20:36'), Timestamp('2026-02-16 23:58:48'))

In [82]:
night_sensor_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 648 entries, 0 to 647
Data columns (total 7 columns):
 #   Column                                                     Non-Null Count  Dtype         
---  ------                                                     --------------  -----         
 0   DateTime                                                   648 non-null    datetime64[ns]
 1   Atlas pH (CH0, Ion Concentration, pH)                      410 non-null    float64       
 2   Atlas CO2 (Carbon Dioxide Gas) (CH0, Carbon Dioxide, ppm)  526 non-null    float64       
 3   Atlas EC (CH0, Electrical Conductivity, μS/cm)             477 non-null    float64       
 4   BME280 (CH0, Temperature, °C)                              495 non-null    float64       
 5   BME280 (CH1, Humidity, %)                                  526 non-null    float64       
 6   BME280 (CH5, Vapor Pressure Deficit, Pa)                   454 non-null    float64       
dtypes: datetime64[ns](1), float64(6)
mem

In [83]:
nutrient_temp_sensor_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 632 entries, 0 to 631
Data columns (total 2 columns):
 #   Column                                Non-Null Count  Dtype         
---  ------                                --------------  -----         
 0   DateTime                              632 non-null    datetime64[ns]
 1   Atlas PT-1000 (CH0, Temperature, °C)  473 non-null    float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 10.0 KB


In [84]:
# concatenate day and night sensor data
sensor_data = pd.concat([day_sensor_data, night_sensor_data], ignore_index=True)
sensor_data.drop(columns=['Atlas pH (CH0, Ion Concentration, pH)',
                          'BME280 (CH5, Vapor Pressure Deficit, Pa)'], inplace=True)
sensor_data.rename(columns={'DateTime':'time_stamp',
                            'Atlas CO2 (Carbon Dioxide Gas) (CH0, Carbon Dioxide, ppm)': 'carbon_dioxide',
                          'Atlas EC (CH0, Electrical Conductivity, μS/cm)': 'electrical_conductivity',
                          'BME280 (CH0, Temperature, °C)': 'temp_env',
                          'BME280 (CH1, Humidity, %)': 'humidity'}, inplace=True)



sensor_data

,time_stamp,carbon_dioxide,electrical_conductivity,temp_env,humidity
0,2026-01-19 19:20:36,0.27,1376.0,22.342666,57.788509
1,2026-01-19 19:23:20,0.27,1468.0,22.596074,53.182916
2,2026-01-19 19:26:04,0.28,1457.5,22.380677,56.845071
3,2026-01-19 19:28:48,0.27,1460.0,22.545392,53.419342
4,2026-01-19 19:31:32,0.28,1465.0,22.246371,56.459571
...,...,...,...,...,...
8537,2026-02-17 11:44:15,511.00,1473.0,23.019267,66.773315
8538,2026-02-17 12:47:32,509.00,1462.0,22.814006,65.947490
8539,2026-02-17 13:50:49,509.00,1443.0,22.864687,66.168633
8540,2026-02-17 14:54:06,512.00,1435.0,22.631551,64.890036


In [85]:
def get_4_day_bin(entry):
    if entry == 0:
        return 0
    elif entry in {1,2,3,4}:
        return 4
    elif entry in {5,6,7,8}:
        return 8
    elif entry in {9,10,11,12}:
        return 12
    elif entry in {13,14,15,16}:
        return 16
    elif entry in {17,18,19,20}:
        return 20
    elif entry in {21,22,23,24}:
        return 24
    elif entry in {25,26,27,28}:
        return 28
    

def add_day_time_bins(df, time_col='time_stamp'):
    df = df.copy()
    day_0_day = df[time_col].dt.floor('D').min()  # Get the start day
    df['temp_day'] = (df[time_col].dt.floor('D') - day_0_day).dt.days
    df['day_period'] = df['temp_day'].apply(get_4_day_bin)
    df.drop(columns=['temp_day'], inplace=True)

    # drop rows with day_period nan (since they are outside 28 day period)
    df.dropna(subset=['day_period'], inplace=True)
  
    return df

sensor_data = add_day_time_bins(sensor_data)

sensor_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8526 entries, 0 to 8525
Data columns (total 6 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   time_stamp               8526 non-null   datetime64[ns]
 1   carbon_dioxide           6162 non-null   float64       
 2   electrical_conductivity  8042 non-null   float64       
 3   temp_env                 5437 non-null   float64       
 4   humidity                 6228 non-null   float64       
 5   day_period               8526 non-null   float64       
dtypes: datetime64[ns](1), float64(5)
memory usage: 466.3 KB


1    electrical_conductivity
3                   humidity
0             carbon_dioxide
2                   temp_env
Name: index, dtype: object

In [102]:
sensor_data[missing_features_sorted]

,electrical_conductivity,humidity,carbon_dioxide,temp_env
0,1376.0,57.788509,0.27,22.342666
1,1468.0,53.182916,0.27,22.596074
2,1457.5,56.845071,0.28,22.380677
3,1460.0,53.419342,0.27,22.545392
4,1465.0,56.459571,0.28,22.246371
...,...,...,...,...
8521,1464.0,75.004872,513.00,24.146947
8522,1484.0,73.934255,507.00,NaN
8523,1487.0,68.500170,507.00,22.748119
8524,1478.5,91.112959,516.00,19.537487


In [95]:
pd.DataFrame(sensor_data.isna().sum()).reset_index().sort_values(by=0, ascending=True)

,index,0
0,time_stamp,0
5,day_period,0
2,electrical_conductivity,484
4,humidity,2298
1,carbon_dioxide,2364
3,temp_env,3089


In [72]:
sensor_data.columns.drop(['time_stamp', 'day_period'])

Index(['carbon_dioxide', 'electrical_conductivity', 'temp_env', 'humidity'], dtype='object')

In [ ]:
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsRegressor

numeric_features = sensor_data.columns.drop(['time_stamp', 'day_period'])



# standardize data
scaler = StandardScaler()
sensor_data_scaled = sensor_data.copy()
sensor_data_scaled[numeric_features] = scaler.fit_transform(sensor_data_scaled[numeric_features])

# initialize imputer 
knn_regressor = KNeighborsRegressor(n_neighbors=5, weights = 'distance')

# sort feature names by missing values (least missing to most missing)
missing_features_sorted = (sensor_data[numeric_features]
                                        .isna().sum()
                                        .sort_values(ascending=True)
                                        .index
                                        .to_list()
                           )

for feature_to_impute in missing_features_sorted:
    features_to_fit_on = [feat for feat in missing_features_sorted if feat != feature_to_impute]

    X = sensor_data_scaled[features_to_fit_on]
    y = sensor_data_scaled[feature_to_impute]
    knn_regressor.fit(X, y)
    print(features_to_fit_on)

,time_stamp,carbon_dioxide,electrical_conductivity,temp_env,humidity,day_period
0,2026-01-19 19:20:36,-1.943235,-1.907102,0.394579,-0.832318,0.0
1,2026-01-19 19:23:20,-1.943235,0.018420,0.540440,-1.158603,0.0
2,2026-01-19 19:26:04,-1.943196,-0.201341,0.416458,-0.899156,0.0
3,2026-01-19 19:28:48,-1.943235,-0.149017,0.511268,-1.141853,0.0
4,2026-01-19 19:31:32,-1.943196,-0.044369,0.339152,-0.926467,0.0
...,...,...,...,...,...,...
8521,2026-02-16 18:51:43,0.036133,-0.065299,1.433121,0.387379,28.0
8522,2026-02-16 19:55:00,0.012970,0.353293,1.099386,0.311531,28.0
8523,2026-02-16 20:58:17,0.012970,0.416082,0.627957,-0.073448,28.0
8524,2026-02-16 22:01:34,0.047714,0.238180,-1.220078,1.528561,28.0


In [ ]:
#DELETE
# impute missing values

from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsRegressor

numeric_features = sensor_data.columns.drop(['time_stamp', 'day_period'])



# standardize data
scaler = StandardScaler()
sensor_data_scaled = sensor_data.copy()
sensor_data_scaled = scaler.fit_transform(sensor_data_scaled[])

# initialize imputer 
knn_regressor = KNeighborsRegressor(n_neighbors=5, weights = 'distance')

# sort feature names by missing values (least missing to most missing)
missing_features_sorted = (sensor_data[numeric_features]
                                        .isna().sum()
                                        .sort_values(ascending=True)
                                        .index
                                        .to_list()
                           )

for feature_to_impute in missing_features_sorted:
    features_to_fit_on = [feat for feat in missing_features_sorted if feat != feature_to_impute]

    X = sensor_data_scaled[features_to_fit_on]
    y = [feature_to_impute]
    sensor_data_scaled[feature_to_impute] = knn_regressor.fit(X, y)
    print(features_to_fit_on)


for feature in missing_features_sorted: 
    current_feature_to_impute = feature
    features_to_fit_on = missing_features_sorted.drop(feature)
    print(features_to_fit_on)
knn_regressor[numeric_features] = knn_regressor.fit([numeric_features])
pipeline = Pipeline([
    ("scaler",  scaler),
    ('knn regressor', knn_regressor)
]
)



sensor_data.iloc[:, 1:] = imputer.fit_transform(sensor_data.iloc[:,1:])
sensor_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8526 entries, 0 to 8525
Data columns (total 6 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   time_stamp               8526 non-null   datetime64[ns]
 1   carbon_dioxide           8526 non-null   float64       
 2   electrical_conductivity  8526 non-null   float64       
 3   temp_env                 8526 non-null   float64       
 4   humidity                 8526 non-null   float64       
 5   day_period               8526 non-null   float64       
dtypes: datetime64[ns](1), float64(5)
memory usage: 466.3 KB


save csv 

In [ ]:
missing_features_sorted = (sensor_data[numeric_features]
                                        .isna().sum()
                                        .sort_values(ascending=True)
                                        .index
                                        .to_list()
                           )

for feature_to_impute in missing_features_sorted:
    features_to_fit_on = [feat for feat in missing_features_sorted if feat != feature_to_impute]
    
    print(features_to_fit_on)


['humidity', 'carbon_dioxide', 'temp_env']
['electrical_conductivity', 'carbon_dioxide', 'temp_env']
['electrical_conductivity', 'humidity', 'temp_env']
['electrical_conductivity', 'humidity', 'carbon_dioxide']


In [106]:
 copy_of_missing_features_sorted = missing_features_sorted.copy()
copy_of_missing_features_sorted

1    electrical_conductivity
3                   humidity
0             carbon_dioxide
2                   temp_env
Name: index, dtype: object